In [ ]:
# 1. Setup: the tree, Bazelisk, the library, and both stage executables.
#
# Colab has no checkout and no build, so this cell clones the tree, runs
# tools/kaggle/bootstrap.sh (Bazelisk, Node.js, the image CUDA toolkit, and the
# host contract), prepares the session, and builds the library with both stage
# executables. Two calls, not one: setup resolves the library only
# (docs/pipeline-plan.md section 6.2).
#
# prefer_installed=False is required, not a preference. setup prefers an
# installed distribution by default, and that root has no tools/nanochat and no
# stage binary, so the wrapped entry of D10 and both children of the chain
# would be missing (D8).
import os
import subprocess
import sys

REPO = "/content/nanochat.cpp"
URL = "https://github.com/ming6ao/nanochat.cpp"

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "notebooks"))

# The bootstrap installs Bazelisk and writes the host contract. setup runs the
# same script on the source path; a second run is safe and fast.
bootstrap = os.path.join(REPO, "tools", "kaggle", "bootstrap.sh")
subprocess.run(["bash", bootstrap], check=True)

import colab_setup
from colab_setup import probe

host = colab_setup.setup(backend="cuda", arch=probe(), prefer_installed=False)
library, sft_main, rl_worker = colab_setup.stages(backend="cuda", arch=probe())
print("repo:", host.repo, "mode:", host.mode)
print("library:", library)
print("stages:", sft_main, rl_worker)

In [ ]:
# 2. Environment: the staging root, the run root, the backend, and the sandbox.
#
# NANOCHAT_BASE_DIR is where the pipeline publishes the tokenizer the RL bridge
# reads. NANOCHAT_RUNS_DIR is the run root, so the job file carries no host path
# and one file runs on all three hosts (D10).
#
# NANOCHAT_SANDBOX_BACKEND=none satisfies the library check in nanochat_init
# only. Both stage executables call RequireSandboxOrDie, which the tool entry of
# the next cell satisfies, and no host may satisfy that check with the override
# (D10): docs/sandbox.md and docs/host-portability.md forbid it, and a Colab
# container has no systemd, so the container quota is the only bound.
import os
import shutil

from colab_setup import probe

RUNS = "/content/runs"
JOB = f"{RUNS}/job.json"

os.environ["NANOCHAT_BASE_DIR"] = "/content/cache"
os.environ["NANOCHAT_RUNS_DIR"] = RUNS
os.environ["NANOCHAT_CPP_BACKEND"] = "cuda"
os.environ["NANOCHAT_CUDA_ARCH"] = probe()
os.environ["NANOCHAT_SANDBOX_BACKEND"] = "none"
os.environ["NANOCHAT_CPP_LIB"] = str(library)

# The tool replaces itself with the command it wraps, so the tree's `python/`
# directory must reach the child through PYTHONPATH: `python -m
# nanochat_cpp.pipeline` starts in this directory and finds no package there.
TREE_PYTHON = os.path.join(REPO, "python")
inherited = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = os.pathsep.join(
    [TREE_PYTHON] + ([inherited] if inherited else []))

# The committed example job holds Job()'s defaults and an empty out_root, so the
# run root comes from NANOCHAT_RUNS_DIR and the file stays host-independent.
os.makedirs(RUNS, exist_ok=True)
shutil.copyfile(f"{REPO}/notebooks/pipeline_job.json", JOB)
print(open(JOB, encoding="utf-8").read())

In [ ]:
# 3. Run: one job file, three stages (base, then SFT, then RL).
#
# The entry is the tool and not `python -m`, because both stage executables call
# RequireSandboxOrDie: the tool sets the sentinel to `off` on a host without
# systemd, and a bare module would start the chain and then fail inside the
# first child (D10, section 6.2).
import shlex
import subprocess

COMMAND = "tools/nanochat train -- python -m nanochat_cpp.pipeline /content/runs/job.json"
print(COMMAND)
subprocess.run(shlex.split(COMMAND), cwd=REPO, check=True)

In [ ]:
# 4. Collect: print the report and offer one download of deliver/.
#
# The report is the review artifact of D6: the stage order and step counts, the
# loss traces and rewards, the checkpoint digests and metadata blocks, the
# seconds and seeds, the model shape, and the source revision. deliver/ holds
# the RL checkpoint and that report.
import json
import os
import shutil
from pathlib import Path

job = json.loads(Path(JOB).read_text(encoding="utf-8"))
deliver = Path(os.environ["NANOCHAT_RUNS_DIR"]) / job["run_id"] / "deliver"
report = json.loads((deliver / "pipeline_report.json").read_text(encoding="utf-8"))

print("run:", job["run_id"], "profile:", report["job"]["profile"])
print("host:", report["host"])
print("total seconds:", round(report["total_seconds"], 2))
for stage in report["stages"]:
    last = stage["losses"][-1] if stage["losses"] else float("nan")
    print(f"{stage['name']}: steps={stage['steps']} last_loss={last:.6f}")
print("delivered:", report["delivered"])

# One download of the whole directory: the checkpoint and the report together.
archive = shutil.make_archive("/content/deliver", "zip", deliver)
try:
    from google.colab import files
except ImportError:
    print("deliver/:", deliver, "archived at:", archive)
else:
    files.download(archive)